In [10]:
import asyncio
from asyncua import Client

class MettlerToledoProbe():
    def __init__(self,opc_url,device_type=''):
        self.device_type         = device_type
        self.opc_url             = opc_url
        self.device_group        = 'MettlerToledoProbe'
        self.if_ready_to_be_used = False
        self.error_msg           = 'No error'
        self.msg                 = {'opc_server_reachable'     : False, 
                                    'hardware_conn': False,
                                    'ready_to_use' : False,
                                    'msg'          : 'data not fetched yet',
                                    'error'        : ''   }
        
        
    async def ping(self):
        try:
            self.msg = await self.fetch_connectivity_status()    
        except Exception as e:
            self.msg['error'] = f'Error: {e}'
        return self.msg

    async def fetch_connectivity_status(self):
        async with Client(url=self.opc_url) as client:
            
            try:
                node_id_opc_server_reachable = f'ns=2;s=Local.{self.device_type}.ConnectionStatus' 
                opc_server_reachable_node = client.get_node(node_id_opc_server_reachable)
                opc_value = await opc_server_reachable_node.read_data_value()
                opc_value = opc_value.Value.Value
                
                if self.device_type in ['iCVision','iCIR']:
                    
                    node_id_hardware_connection = f'ns=2;s=Local.{self.device_type}.Probe1.ProbeStatus' 
                    hardware_connection_node = client.get_node(node_id_hardware_connection)
    
                    try:
                        hardware_value = await hardware_connection_node.read_data_value()
                        hardware_value = hardware_value=='Running'
                        
                    except Exception as e:
                        hardware_value = False
                        
                        if 'BadOutOfService' in str(e):
                            self.error_msg = f'Error: {e}'
                        else:
                            self.error_msg = f'Unknown error: {e}'
             
                elif self.device_type in ['iCFBRM']:
    
                    node_id_hardware_connection = f'ns=2;s=Local.{self.device_type}.InstrumentDescription' 
                    hardware_connection_node = client.get_node(node_id_hardware_connection)
                    try:
                        hardware_value = await hardware_connection_node.read_value()
                        hardware_value = '<instrumemt not available>' not in hardware_value
                    except Exception as e:
                        hardware_value = False
                        self.error_msg = f'Error: {e}'
    
                return  {'opc_server_reachable' : opc_value, 
                         'hardware_conn'        : hardware_value,
                         'ready_to_use'         : opc_value and hardware_value,
                         'msg'                  : f'OPC:{opc_value}, Hardware:{hardware_value}, if_ready:{opc_value and hardware_value}',
                         'error'                : self.error_msg }
                
            except Exception as e:
                self.error_msg = f'Error {e}'
                return  {'opc_server_reachable' : False, 
                         'hardware_conn'        : False,
                         'ready_to_use'         : False,
                         'msg'                  : f'OPC:{opc_value}, Hardware:{hardware_value}, if_ready:{opc_value and hardware_value}',
                         'error'                : self.error_msg     }
        

class ComponentConnectivity():
    def __init__(self,Device):
        self.device_group = Device.device_group 
        self.device_type  = Device.device_type
        self.Device       = Device
        
    async def check_connectivity(self):
        self.msg = await self.Device.ping()
        
        if self.msg['ready_to_use']:
            self.connected = True
            return self.connected, self.msg
        else:
            self.connected = False
            return self.connected, self.msg    
        

In [11]:
opc_url = "opc.tcp://localhost:62552/iCOpcUaServer"
IR   = MettlerToledoProbe(opc_url,'iCIR')
FBRM = MettlerToledoProbe(opc_url,'iCFBRM')
EZV  = MettlerToledoProbe(opc_url,'iCVision')

IR_conn = ComponentConnectivity(IR)
FBRM_conn = ComponentConnectivity(FBRM)
EZV_conn = ComponentConnectivity(EZV)

for conn in [IR_conn, FBRM_conn, EZV_conn]:
    status, msg = await conn.check_connectivity()
    print(f'Device type: {conn.Device.device_type}, Connected?--> {status}')



Device type: iCIR, Connected?--> False
Device type: iCFBRM, Connected?--> False
Device type: iCVision, Connected?--> False


In [7]:
print(FBRM_conn.msg)


{'opc_server_reachable': False, 'hardware_conn': False, 'ready_to_use': False, 'msg': 'data not fetched yet', 'error': 'Error: '}
